# 01 - Fundamentos de Regularización y la Importancia del Escalado

## Curso de Machine Learning: Modelos de Regresion

En este cuaderno estudiaremos los fundamentos teoricos que motivan las tecnicas de regularizacion en aprendizaje automatico. Analizaremos por que el metodo clasico de Minimos Cuadrados Ordinarios (OLS) es vulnerable ante problemas de multicolinealidad y alta varianza, definiremos formalmente las normas matematicas $L_1$ y $L_2$, y demostraremos por que el escalado de variables es un requisito indispensable antes de aplicar cualquier penalizacion.

---

### Contenidos
1. Limitaciones de Minimos Cuadrados Ordinarios (OLS)
2. El Dilema Sesgo-Varianza (Bias-Variance Tradeoff)
3. Normas Matematicas: Norma $L_1$ vs. Norma $L_2$
4. Por que el Escalado es Obligatorio en Regularizacion
5. Exploracion del Dataset Diabetes (`load_diabetes`)
6. Evaluacion de Multicolinealidad
7. Demostracion Practica: OLS con y sin Escalado en Pipelines
8. Conclusiones y Siguientes Pasos


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Limitaciones de Minimos Cuadrados Ordinarios (OLS)

En regresion lineal clasica, buscamos el vector de coeficientes $\hat{\beta}$ que minimiza la suma de errores cuadraticos:

$$J(\beta) = \frac{1}{2n} \sum_{i=1}^n (y_i - \hat{y}_i)^2 = \frac{1}{2n} \|y - X\beta\|_2^2$$

La solucion analitica cerrada viene dada por la ecuacion normal:

$$\hat{\beta} = (X^T X)^{-1} X^T y$$

Sin embargo, este estimador presenta dos problemas criticos en la practica:
1. **Multicolinealidad**: Si dos o mas variables predictoras estan fuertemente correlacionadas, la matriz $X^T X$ se aproxima a una matriz singular (determinante cercano a 0). Esto produce una inversion numericamente inestable, donde pequenisimos cambios en los datos de entrenamiento causan fluctuaciones gigantescas en los valores de los coeficientes.
2. **Alta Dimensionalidad ($p \approx n$ o $p > n$)**: Si el numero de parametros $p$ es grande en relacion al numero de muestras $n$, el modelo memoriza el ruido de entrenamiento, generando sobreajuste (overfitting) y una pobre capacidad de generalizacion en datos no observados.


## 2. El Dilema Sesgo-Varianza (Bias-Variance Tradeoff)

El error esperado de generalizacion de un modelo predictivo se descompone en tres componentes:

$$\mathbb{E}[(y - \hat{f}(x))^2] = \text{Sesgo}^2(\hat{f}(x)) + \text{Varianza}(\hat{f}(x)) + \sigma^2$$

- **Sesgo (Bias)**: Error introducido al aproximar un problema real complejo mediante un modelo simplificado. OLS tiene bajo sesgo (de hecho, es insesgado bajo el Teorema de Gauss-Markov).
- **Varianza (Variance)**: Cuanto cambiaria la prediccion $\hat{f}(x)$ si la entrenaramos con un conjunto de datos diferente. OLS tiende a tener una varianza muy alta cuando hay muchas variables o multicolinealidad.
- **Error irreducible ($\sigma^2$)**: Ruido intrinseco e inevitable de los datos.

**Idea central de la regularizacion:**
Aceptar introducir un pequeno **sesgo** controlado a cambio de conseguir una reduccion sustancial de la **varianza**, logrando que el error total de generalizacion sea notablemente menor.


## 3. Normas Matematicas: Norma $L_1$ vs. Norma $L_2$

Para limitar la complejidad del modelo, la regularizacion anade una penalizacion basada en el tamano del vector de parametros $\beta = [\beta_1, \beta_2, \dots, \beta_p]^T$.

### Norma $L_1$ (Manhattan)
Mide la suma de las magnitudes absolutas de los coeficientes:

$$\|\beta\|_1 = \sum_{j=1}^p |\beta_j|$$

- Es la base de la regresion **Lasso**.
- Genera soluciones dispersas (sparse), forzando a que algunos coeficientes sean exactamente cero.

### Norma $L_2$ (Euclidiana)
Mide la longitud vectorial geometrica (o su version al cuadrado):

$$\|\beta\|_2 = \sqrt{\sum_{j=1}^p \beta_j^2}, \quad \|\beta\|_2^2 = \sum_{j=1}^p \beta_j^2$$

- Es la base de la regresion **Ridge**.
- Reduce los coeficientes hacia valores pequenos y estables, pero no los anula por completo.

A continuacion, visualizaremos geometricamente las curvas de nivel de ambas normas en un espacio bidimensional $(\beta_1, \beta_2)$.


In [ ]:
# Visualizacion geometrica de las normas L1 y L2
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Espacio de parametros
b1 = np.linspace(-2, 2, 400)
b2 = np.linspace(-2, 2, 400)
B1, B2 = np.meshgrid(b1, b2)

# Norma L1 y L2
L1 = np.abs(B1) + np.abs(B2)
L2 = np.sqrt(B1**2 + B2**2)

# Grafico Norma L1
cs1 = axes[0].contour(B1, B2, L1, levels=[0.5, 1.0, 1.5], colors=['#2b5c8f', '#1f77b4', '#0d3d63'], linewidths=2)
axes[0].clabel(cs1, inline=True, fontsize=10)
axes[0].set_title('Contornos de Norma L1: Rombo (Lasso)', fontsize=12, fontweight='bold')
axes[0].set_xlabel(r'$\beta_1$', fontsize=11)
axes[0].set_ylabel(r'$\beta_2$', fontsize=11)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.6)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.6)
axes[0].set_aspect('equal')

# Grafico Norma L2
cs2 = axes[1].contour(B1, B2, L2, levels=[0.5, 1.0, 1.5], colors=['#8c2d19', '#d62728', '#59160a'], linewidths=2)
axes[1].clabel(cs2, inline=True, fontsize=10)
axes[1].set_title('Contornos de Norma L2: Circulo (Ridge)', fontsize=12, fontweight='bold')
axes[1].set_xlabel(r'$\beta_1$', fontsize=11)
axes[1].set_ylabel(r'$\beta_2$', fontsize=11)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.6)
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.6)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()


## 4. Por que el Escalado es Obligatorio en Regularizacion

En Minimos Cuadrados Ordinarios (sin regularizacion), si multiplicamos una variable $X_j$ por una constante $c$, su coeficiente $\beta_j$ simplemente se divide por $c$. Las predicciones $\hat{y}$ permanecen invariantes.

**En modelos regularizados esto NO ocurre:**
La funcion de costo suma el error cuadratico mas una penalizacion uniforme sobre todos los coeficientes:

$$J_{reg}(\beta) = \text{Error}(y, X\beta) + \alpha \sum_{j=1}^p \text{Penalizacion}(\beta_j)$$

- Si una variable esta medida en millones (por ejemplo, ingresos anuales), su coeficiente natural sera muy pequeno (ej. $0.00003$). Su penalizacion sera insignificante.
- Si otra variable esta medida en decimas (por ejemplo, concentracion quimica de $0.1$ a $0.9$), su coeficiente sera grande (ej. $150.0$). Su penalizacion sera descomunal.
- **Consecuencia:** Sin estandarizar, la regularizacion penalizara injustamente a las variables con escalas pequenas y dejara desatendidas a las variables con escalas grandes.

### Regla de Oro
1. Todas las caracteristicas deben ser estandarizadas con media cero y varianza unitaria (Z-Score):
   $$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$
2. **El intercepto $\beta_0$ NUNCA se penaliza.** El sesgo global de la variable objetivo no debe restringirse hacia cero.


## 5. Exploracion del Dataset Diabetes (`load_diabetes`)

Utilizaremos el dataset clasico de diabetes de Efron et al. (2004), ampliamente reconocido en la literatura estadistica para evaluar metodos de regularizacion.

- **Muestras**: 442 pacientes diagnosticados con diabetes.
- **Variables predictoras (10 caracteristicas)**:
  - `age`: Edad del paciente.
  - `sex`: Genero (1 = mujer, 2 = hombre en formato crudo).
  - `bmi`: Indice de masa corporal.
  - `bp`: Presion arterial promedio.
  - `s1` a `s6`: Seis mediciones clinicas de suero sanguineo (colesterol total, ldl, hdl, tch, ltg, glucosa).
- **Variable objetivo (`target`)**: Medida cuantitativa de la progresion de la enfermedad un ano despues de la linea base.


In [ ]:
# Carga del dataset en formato crudo (sin pre-escalado de sklearn)
diabetes_raw = load_diabetes(scaled=False, as_frame=True)
df = diabetes_raw.frame

print("Dimensiones del dataset:", df.shape)
print("\nPrimeras 5 observaciones:")
df.head()


In [ ]:
# Estadisticas descriptivas para observar la disparidad de escalas
resumen_escalas = df.describe().T[['mean', 'std', 'min', 'max']]
resumen_escalas.columns = ['Media', 'Desv. Estandar', 'Minimo', 'Maximo']
print("Comparativa de escalas entre caracteristicas:")
resumen_escalas.round(2)


## 6. Evaluacion de Multicolinealidad

Analizaremos la matriz de correlacion de Pearson. Prestaremos especial atencion a las variables de suero sanguineo `s1` a `s6`, que frecuentemente exhiben fuertes dependencias lineales entre si.


In [ ]:
# Matriz de correlacion de las caracteristicas
X_raw = df.drop(columns=['target'])
y = df['target']

corr_matrix = X_raw.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, cbar_kws={'label': 'Coeficiente de Correlacion'})
plt.title('Matriz de Correlacion de Caracteristicas Predictoras', fontsize=13, fontweight='bold')
plt.show()


In [ ]:
# Identificacion automatica de pares con alta correlacion (|r| > 0.70)
pares_colineales = []
cols = corr_matrix.columns

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        val = corr_matrix.iloc[i, j]
        if abs(val) >= 0.70:
            pares_colineales.append((cols[i], cols[j], val))

print("Pares de variables con alta multicolinealidad (|r| >= 0.70):")
for v1, v2, r in pares_colineales:
    print(f"- {v1} y {v2}: r = {r:.3f}")


## 7. Demostracion Practica: OLS con y sin Escalado en Pipelines

Entrenaremos un modelo OLS tradicional utilizando Scikit-Learn:
1. Sin estandarizar las variables predictoras.
2. Estandarizando las variables mediante `StandardScaler` dentro de un `Pipeline`.

Observaremos como las predicciones y metricas de OLS no cambian, pero los coeficientes si cambian para reflejar la importancia relativa de cada variable normalizada.


In [ ]:
# Particion de datos en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=42
)

# 1. OLS sin escalado
modelo_ols_sin_escala = LinearRegression()
modelo_ols_sin_escala.fit(X_train, y_train)

# 2. OLS con escalado dentro de un Pipeline
pipeline_ols_escalado = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])
pipeline_ols_escalado.fit(X_train, y_train)

# Predicciones en conjunto de prueba
y_pred_sin = modelo_ols_sin_escala.predict(X_test)
y_pred_con = pipeline_ols_escalado.predict(X_test)

# Comparativa de metricas de desempeno
print("Metricas OLS sin escalado:")
print(f"  MSE:  {mean_squared_error(y_test, y_pred_sin):.2f}")
print(f"  R2:   {r2_score(y_test, y_pred_sin):.4f}")

print("\nMetricas OLS con escalado (StandardScaler):")
print(f"  MSE:  {mean_squared_error(y_test, y_pred_con):.2f}")
print(f"  R2:   {r2_score(y_test, y_pred_con):.4f}")


In [ ]:
# Comparativa de magnitudes de coeficientes
coef_sin = modelo_ols_sin_escala.coef_
coef_con = pipeline_ols_escalado.named_steps['regressor'].coef_

df_coef = pd.DataFrame({
    'Variable': X_raw.columns,
    'Coef_Sin_Escalar': coef_sin,
    'Coef_Estandarizado': coef_con
})

print("Impacto del escalado en los coeficientes:")
print(df_coef.round(3).to_string(index=False))

# Visualizacion comparativa
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(df_coef['Variable'], df_coef['Coef_Sin_Escalar'], color='#4a7c59')
axes[0].set_title('Coeficientes OLS (Datos sin escalar)\nReflejan unidades de medicion', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Valor del Coeficiente')
axes[0].axvline(0, color='black', linestyle='--', alpha=0.5)

axes[1].barh(df_coef['Variable'], df_coef['Coef_Estandarizado'], color='#2b5c8f')
axes[1].set_title('Coeficientes OLS (Estandarizados con StandardScaler)\nComparables directamente en importancia', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Valor del Coeficiente Estandarizado')
axes[1].axvline(0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


## 8. Conclusiones y Siguientes Pasos

### Puntos Clave Aprendidos:
1. **OLS es sensible a multicolinealidad y sobreajuste**: Cuando hay correlaciones elevadas entre predictores (como observamos en `s1`, `s2` y `s4`), los estimadores analiticos son numericamente fragiles.
2. **Normas $L_1$ y $L_2$**: Proporcionan geometrias de penalizacion fundamentalmente distintas (rombo vs. circulo) que daran origen a Lasso y Ridge.
3. **Escalado previo estricto**: En OLS las predicciones no cambian al escalar, pero en los modelos regularizados, no escalar arruina el balance de la penalizacion.
4. **Construccion con `Pipeline`**: Siempre debemos encapsular `StandardScaler` junto al estimador para evitar fuga de informacion (data leakage) del conjunto de prueba.

En el siguiente cuaderno, **02_Regresion_Ridge_L2.ipynb**, estudiaremos a profundidad la regularizacion $L_2$, su solucion matricial, el encogimiento de coeficientes y la optimizacion automatica del hiperparametro $\alpha$ con `RidgeCV`.
